# Forest Cover Type Prediction — Kaggle 2

**IDC 6210 | Supervised multiclass classification**

## Research question and modeling plan

The goal is to predict each observation's **forest-cover class** (`label`) from the available geographic and environmental measurements. Because the labels are known for the training records, this is a **supervised classification** problem, not regression. It is *multiclass*: a prediction must identify the correct category, not just choose between two outcomes.

I compare the five algorithms required by the assignment: **Perceptron, Logistic Regression, K-Nearest Neighbors (KNN), Decision Tree, and Random Forest**. The workflow is to inspect the data, separate predictors from labels, create a stratified training/validation split, prepare features without leakage, fit every model, compare their out-of-sample results, and then train the selected approach on all available labeled rows before submitting test predictions.

**Evaluation note:** The supplied assignment template calls for F1 scores but does not state an averaging method. I therefore report **weighted F1, macro F1, and accuracy**. Weighted F1 is the default comparison metric here because it accounts for class frequency; macro F1 reveals weaker performance on less common classes. **If this specific Kaggle competition grades accuracy instead of F1, change `PRIMARY_METRIC` to `"accuracy"` before choosing the final model.** The Kaggle leaderboard score, not the local validation score, determines the competition-performance portion of the grade.

---
# Part 1 — Prepare, train, and validate models

## 1. Import libraries and locate the data

**Why:** Pandas and NumPy handle the dataset, while scikit-learn provides reproducible preprocessing, classification models, and evaluation tools. The Kaggle notebook's attached competition dataset supplies the training and test CSV files.

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

RANDOM_STATE = 42
VALIDATION_FRACTION = 0.20
PRIMARY_METRIC = "weighted_f1"  # Change to "accuracy" if specified by this competition.

# Original competition data are excluded from this public portfolio.
# Works on Kaggle or locally with data/train.csv and data/test.csv.
KAGGLE_DATA_DIR = "/kaggle/input/competitions/forest-cover-type-prediction-fall-2026-idc-6210"
DATA_DIR = os.environ.get(
    "FOREST_COVER_DATA_DIR",
    KAGGLE_DATA_DIR if os.path.isdir(KAGGLE_DATA_DIR) else "data"
)
if not all(os.path.isfile(os.path.join(DATA_DIR, name)) for name in ("train.csv", "test.csv")):
    raise FileNotFoundError(
        f"Could not find train.csv and test.csv in '{DATA_DIR}'. "
        "Put the original Kaggle competition files in data/ or set FOREST_COVER_DATA_DIR."
    )
print("Using dataset directory:", DATA_DIR)
print("Files:", os.listdir(DATA_DIR))

## 2. Load the training data and identify predictors

The `label` column is the **ground-truth class** that we are trying to learn. The `id` column identifies records but does not describe forest conditions. Neither belongs in the predictor matrix: including `label` would be **target leakage**, and treating an arbitrary identifier as an environmental feature could create misleading patterns.

In [ ]:
df_traindata = pd.read_csv(f"{DATA_DIR}/train.csv")
print("Training data shape (rows, columns):", df_traindata.shape)
display(df_traindata.head())

assert {"id", "label"}.issubset(df_traindata.columns), (
    "Expected 'id' and 'label' columns in the training CSV."
)
X = df_traindata.drop(columns=["id", "label"])
y = df_traindata["label"]

print(f"Predictor columns: {X.shape[1]}")
print(f"Forest-cover classes: {y.nunique()}")
print("All predictors numeric:", X.select_dtypes(exclude="number").empty)
assert X.select_dtypes(exclude="number").empty, "Unexpected nonnumeric predictors require encoding."
display(X.head())
display(y.head().to_frame())

## 3. Inspect data quality and class balance

**Why:** Checking data types, missing values, duplicate IDs, and class counts determines what preprocessing is needed and whether one class dominates the training sample. A high overall accuracy can hide poor predictions for a minority class, so the class distribution matters when interpreting F1 results.

In [ ]:
print("Data types:")
display(X.dtypes.value_counts().rename("number_of_columns").to_frame())
print("Missing values by column (nonzero only):")
missing = X.isna().sum()
display(missing[missing > 0].sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate training IDs:", df_traindata["id"].duplicated().sum())
print("Duplicate full training rows:", df_traindata.duplicated().sum())
print("Target class distribution:")
class_summary = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "percentage": (y.value_counts(normalize=True).sort_index() * 100).round(2),
})
display(class_summary)
print("Predictor summary (first 12 columns):")
display(X.describe().T.head(12))

### Data inspection: what the output means

The table above establishes the **observed** feature count, missing-value pattern, and number of examples per cover type; these are checked rather than assumed. If class sizes differ, **macro F1** will be especially useful because it gives each class equal influence. I retain the original labeled observations rather than automatically deleting duplicates, because identical geographic feature patterns are not automatically data errors. No preprocessing statistics will be estimated from the validation set or unlabeled test set.

## 4. Split labeled data into training and validation sets

I reserve **20% for validation** and train on the remaining **80%**. `stratify=y` maintains approximately the same cover-type proportions in both sets, which makes model comparisons more reliable when categories are unevenly represented. `random_state=42` makes the split reproducible.

**Important distinction:** The Kaggle `test.csv` has no true `label`, so it cannot tell us which model performs best. That choice must be made with labeled validation data before generating the submission.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=VALIDATION_FRACTION,
    stratify=y,
    random_state=RANDOM_STATE
)
print("Training features:", X_train.shape, "Validation features:", X_val.shape)
print("Training labels:", y_train.shape, "Validation labels:", y_val.shape)
display(pd.DataFrame({
    "train_pct": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "validation_pct": (y_val.value_counts(normalize=True).sort_index() * 100).round(2),
}))

## 5. Handle missing data without leakage

A **median imputer** learns a replacement value for each numerical feature from the **training split only** and uses those same learned values on validation records. This is safer than dropping many rows and less sensitive to extreme values than mean imputation. Even when the source has no missing values, the consistent procedure protects the model against missing fields in future/test data.

Using `fit_transform()` on validation data would leak information from the holdout set into preprocessing, making the reported validation result less trustworthy.

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X.columns, index=X_train.index
)
X_val_imputed = pd.DataFrame(
    imputer.transform(X_val), columns=X.columns, index=X_val.index
)
print("Missing values after imputation — train:", int(X_train_imputed.isna().sum().sum()))
print("Missing values after imputation — validation:", int(X_val_imputed.isna().sum().sum()))

## 6. Engineer geographic features and scale continuous measurements

### Feature engineering

The environmental measurements may contain **nonlinear interactions**. If the necessary source columns exist, I add the straight-line distance to water using horizontal and vertical hydrology measurements, and the range of sunlight/shade measurements across the day. These derived predictors are plausible summaries of terrain and lighting; **they are hypotheses, not guaranteed improvements**. The same formulas are applied to training, validation, and Kaggle test data.

### Scaling

As discussed in our preprocessing notes, **KNN is distance-based**, while Perceptron and Logistic Regression learn linear weights and are sensitive to feature magnitude. I use `StandardScaler` to standardize **continuous variables** using training-only means and standard deviations. Binary wilderness/soil indicator columns remain 0/1 so that scaling does not inflate rare indicator values. Decision Trees and Random Forests generally do not need scaling because they split at feature thresholds.

For a continuous feature, standardization computes \(z=(x-\mu_{train})/\sigma_{train}\). Only the training split supplies \(\mu\) and \(\sigma\); the validation set uses the already fitted scaler. This is another safeguard against data leakage.

In [ ]:
def add_geographic_features(frame):
    """Row-by-row transformations requiring no target labels or fitted statistics."""
    out = frame.copy()
    if {"Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology"}.issubset(out.columns):
        out["Hydrology_Distance_3D"] = np.hypot(
            out["Horizontal_Distance_To_Hydrology"],
            out["Vertical_Distance_To_Hydrology"]
        )
    shade_cols = ["Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm"]
    if set(shade_cols).issubset(out.columns):
        out["Hillshade_Range"] = out[shade_cols].max(axis=1) - out[shade_cols].min(axis=1)
    return out

X_train_ready = add_geographic_features(X_train_imputed)
X_val_ready = add_geographic_features(X_val_imputed)
assert X_train_ready.columns.tolist() == X_val_ready.columns.tolist()

# Scale non-binary columns, retain existing binary indicator columns in their 0/1 form.
continuous_cols = [col for col in X_train_ready.columns if X_train_ready[col].nunique() > 2]
binary_or_constant_cols = [col for col in X_train_ready.columns if col not in continuous_cols]
scaler = StandardScaler()
X_train_scaled = X_train_ready.copy()
X_val_scaled = X_val_ready.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train_ready[continuous_cols])
X_val_scaled[continuous_cols] = scaler.transform(X_val_ready[continuous_cols])

print("Engineered columns:", [col for col in X_train_ready if col not in X.columns])
print("Continuous columns scaled:", len(continuous_cols))
print("Binary/constant columns preserved:", len(binary_or_constant_cols))
print("Final feature count:", X_train_ready.shape[1])

## 7. Build the five required classifiers

I use the **same training/validation rows** for every candidate so the comparison is fair. Hyperparameters are explicit and the random seed is fixed where supported. All five are classification models, but they make different assumptions about the relationship between terrain and forest type.

| Model | Learning approach | Main expected strength | Main limitation |
|---|---|---|---|
| Perceptron | Learns linear class boundaries | Fast, simple baseline | May underfit complex/nonlinear interactions |
| Logistic Regression | Regularized linear classifier | Stable and interpretable benchmark | Linear decision boundaries without added interactions |
| KNN | Nearby observations vote on the label | Captures local nonlinear structure | Sensitive to distances; expensive on large datasets |
| Decision Tree | Recursively partitions feature values | Captures rules and interactions | A single deep tree may overfit |
| Random Forest | Aggregates many randomized trees | Handles nonlinearities and reduces single-tree variance | More computationally expensive and less transparent |

These expectations will be tested against **actual validation metrics** rather than treated as results.

In [ ]:
pct_model = Perceptron(max_iter=1000, tol=1e-3, random_state=RANDOM_STATE)
log_model = LogisticRegression(max_iter=2000, C=1.0)
knn_model = KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1)
dt_model = DecisionTreeClassifier(random_state=RANDOM_STATE, min_samples_leaf=2)
rf_model = RandomForestClassifier(
    n_estimators=250, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
)

models = {
    "Perceptron": pct_model,
    "Logistic Regression": log_model,
    "KNN": knn_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model,
}
scaled_models = {"Perceptron", "Logistic Regression", "KNN"}
print("Required classifiers:", ", ".join(models))

## 8. Train models on the training split only

The Perceptron, Logistic Regression, and KNN are fitted on scaled features; the Decision Tree and Random Forest use the same imputed/engineered features without scaling. Training **does not access the validation labels**. The tree-based models can learn interactions directly, while the linear models provide a useful indication of whether the forest types are linearly separable in this representation.

In [ ]:
pct_model.fit(X_train_scaled, y_train)
log_model.fit(X_train_scaled, y_train)
knn_model.fit(X_train_scaled, y_train)
dt_model.fit(X_train_ready, y_train)
rf_model.fit(X_train_ready, y_train)
print("All five models fitted successfully.")

## 9. Predict the held-out validation labels

Each model predicts labels for **exactly the same validation observations**. We can now compare each prediction with its known ground truth. For KNN/linear models, the validation features must go through the *training-fitted* scaler; tree models use the unscaled engineered features.

In [ ]:
pct_pred = pct_model.predict(X_val_scaled)
log_pred = log_model.predict(X_val_scaled)
knn_pred = knn_model.predict(X_val_scaled)
dt_pred = dt_model.predict(X_val_ready)
rf_pred = rf_model.predict(X_val_ready)

predictions = {
    "Perceptron": pct_pred,
    "Logistic Regression": log_pred,
    "KNN": knn_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
}

## 10. Evaluate all models using several complementary metrics

**Weighted F1** combines precision and recall within each class, then weights classes by their frequency. It summarizes typical performance while accounting for the number of examples in each category. **Macro F1** gives every class equal weight, revealing whether rare forest types are being overlooked. **Accuracy** is the fraction of observations classified correctly and is useful for understanding overall errors and for any Kaggle leaderboard evaluated by accuracy.

Because this is **multiclass classification**, `f1_score()` needs an explicit averaging rule. I use `average="weighted"` for the five rubric-template F1 values and separately calculate macro F1. A **training–validation accuracy gap** can suggest overfitting: excellent training performance with much weaker validation performance indicates that the model may be memorizing the training rows instead of generalizing. The gap alone does not prove overfitting; it is evidence to interpret alongside absolute validation performance.

In [ ]:
# Required F1 comparisons: explicitly use weighted multiclass averaging.
pct_f1_score = f1_score(y_val, pct_pred, average="weighted", zero_division=0)
log_f1_score = f1_score(y_val, log_pred, average="weighted", zero_division=0)
knn_f1_score = f1_score(y_val, knn_pred, average="weighted", zero_division=0)
dt_f1_score = f1_score(y_val, dt_pred, average="weighted", zero_division=0)
rf_f1_score = f1_score(y_val, rf_pred, average="weighted", zero_division=0)

f1_by_model = {
    "Perceptron": pct_f1_score,
    "Logistic Regression": log_f1_score,
    "KNN": knn_f1_score,
    "Decision Tree": dt_f1_score,
    "Random Forest": rf_f1_score,
}

results = []
for name, fitted_model in models.items():
    x_for_training = X_train_scaled if name in scaled_models else X_train_ready
    training_predictions = fitted_model.predict(x_for_training)
    validation_predictions = predictions[name]
    train_accuracy = accuracy_score(y_train, training_predictions)
    val_accuracy = accuracy_score(y_val, validation_predictions)
    results.append({
        "model": name,
        "train_accuracy": train_accuracy,
        "accuracy": val_accuracy,
        "weighted_f1": f1_by_model[name],
        "macro_f1": f1_score(y_val, validation_predictions, average="macro", zero_division=0),
        "accuracy_gap": train_accuracy - val_accuracy,
    })

model_comparison = pd.DataFrame(results)
assert PRIMARY_METRIC in {"weighted_f1", "macro_f1", "accuracy"}
model_comparison = model_comparison.sort_values(
    [PRIMARY_METRIC, "accuracy"], ascending=False
).reset_index(drop=True)
print("Validation model comparison (higher accuracy/F1 is better):")
display(model_comparison.style.format({
    "train_accuracy": "{:.4f}", "accuracy": "{:.4f}",
    "weighted_f1": "{:.4f}", "macro_f1": "{:.4f}",
    "accuracy_gap": "{:+.4f}"
}))

## 11. Interpret the measured comparison and select a model

**Decision rule:** I first rank the candidates by the chosen primary validation metric. I also inspect **macro F1** (whether weaker classes are being missed), **accuracy** (overall classification), and the **training–validation gap** (possible overfitting). The strongest model is not necessarily the most complex one: a simpler model is preferable if its validation performance is comparable and its speed or interpretability is substantially better. The final choice is made **before** looking at any Kaggle test labels, which are unavailable.

The following cell creates a numerical interpretation **from the actual validation table** so the discussion is specific rather than inventing results. It compares the top two models and examines class-level weaknesses. Review the generated findings and use them in your final submitted Markdown discussion.

In [ ]:
best_name = model_comparison.loc[0, "model"]
runner_up_name = model_comparison.loc[1, "model"]
best_row = model_comparison.iloc[0]
runner_row = model_comparison.iloc[1]
best_model = models[best_name]
best_pred = predictions[best_name]
score_difference = best_row[PRIMARY_METRIC] - runner_row[PRIMARY_METRIC]

strengths = {
    "Perceptron": "a fast, simple linear baseline",
    "Logistic Regression": "a regularized and relatively interpretable linear model",
    "KNN": "a locally adaptive model that can capture nonlinear boundaries",
    "Decision Tree": "interpretable threshold-based rules and interactions",
    "Random Forest": "an ensemble of trees that captures nonlinear patterns and reduces individual-tree variance",
}
limitations = {
    "Perceptron": "linear boundaries can miss complex ecological interactions",
    "Logistic Regression": "without richer interaction terms, its boundaries remain linear",
    "KNN": "predictions depend on distance choices and can become slow as data volume grows",
    "Decision Tree": "a single tree can be unstable and especially prone to overfitting",
    "Random Forest": "the ensemble is harder to interpret and costs more to train and predict with",
}

class_report = classification_report(y_val, best_pred, output_dict=True, zero_division=0)
class_rows = []
for label_value in sorted(y_val.unique()):
    report_row = class_report[str(label_value)]
    class_rows.append((label_value, report_row["f1-score"], int(report_row["support"])))
weakest_class, weakest_f1, weakest_support = min(class_rows, key=lambda item: item[1])

if best_row["accuracy_gap"] > 0.10:
    gap_interpretation = "A sizable training–validation accuracy gap suggests possible overfitting."
elif best_row["accuracy_gap"] < -0.02:
    gap_interpretation = "Validation accuracy exceeds training accuracy; this can reflect sampling variability and merits a closer look."
else:
    gap_interpretation = "The train–validation accuracy gap is relatively small, which is reassuring but not proof of generalization."

comparison_note = (
    "The scores are close, so model stability, class-level results and practical complexity deserve attention."
    if score_difference < 0.01 else
    "The difference provides validation-set evidence in favor of the selected model, although it may vary with another split."
)

display(Markdown(f"""
### Results-based model discussion

**Model comparison:** The highest **{PRIMARY_METRIC.replace('_', ' ')}** is from **{best_name}**
(**{best_row[PRIMARY_METRIC]:.4f}**), followed by **{runner_up_name}**
(**{runner_row[PRIMARY_METRIC]:.4f}**). The advantage is **{score_difference:.4f}**.
The selected model's validation accuracy is **{best_row['accuracy']:.4f}**, weighted F1 is
**{best_row['weighted_f1']:.4f}**, and macro F1 is **{best_row['macro_f1']:.4f}**.
{comparison_note}

**Model selection:** I selected **{best_name}** because it had the strongest measured validation
**{PRIMARY_METRIC.replace('_', ' ')}** of the five required models. Its practical strength is
**{strengths[best_name]}**. Compared with **{runner_up_name}**, it performed better on the
selected validation metric; however, **{limitations[best_name]}**.

**Generalization:** Training accuracy is **{best_row['train_accuracy']:.4f}** and validation
accuracy is **{best_row['accuracy']:.4f}**, a difference of
**{best_row['accuracy_gap']:+.4f}**. {gap_interpretation}

**Class-level caution:** Class **{weakest_class}** has the lowest validation F1 (**{weakest_f1:.4f}**;
**{weakest_support}** validation examples). That limits the claim that the model performs equally
well on every cover type. A single 80/20 split also introduces sampling uncertainty; the
unlabeled Kaggle test set cannot verify this score.
"""))

## 12. Examine class-level errors and model behavior

The classification report separates **precision**, **recall**, and **F1** for each forest-cover type, while the confusion matrix shows which categories are mistaken for one another. These diagnostics matter because two models can share similar overall F1 yet behave differently on difficult classes. Feature rankings, when available, describe patterns used by a fitted tree or linear model; they do **not** establish that a geographic variable causes a particular forest type.

In [ ]:
print("Per-class classification report for:", best_name)
print(classification_report(y_val, best_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_val, best_pred, normalize="true", ax=ax, cmap="Blues", values_format=".2f"
)
ax.set_title(f"{best_name} — normalized validation confusion matrix")
plt.tight_layout()
plt.show()

if hasattr(best_model, "feature_importances_"):
    feature_strength = pd.Series(
        best_model.feature_importances_, index=X_train_ready.columns
    ).sort_values(ascending=False)
    print("Most-used features in fitted tree splits (importance, not causality):")
    display(feature_strength.head(10).to_frame("feature_importance"))
elif hasattr(best_model, "coef_"):
    feature_strength = pd.Series(
        np.abs(best_model.coef_).mean(axis=0), index=X_train_ready.columns
    ).sort_values(ascending=False)
    print("Largest average absolute linear coefficients (not causal effects):")
    display(feature_strength.head(10).to_frame("mean_abs_coefficient"))
else:
    print("This estimator has no direct feature-importance attribute; class-level metrics remain available.")

---
# Part 2 — Predict the unlabeled Kaggle test set

## 13. Prepare the test data using the identical feature definitions

The Kaggle test set has no labels. I remove the `id` field from its predictors while retaining it for submission. I verify that its original feature columns match the training data, then **refit preprocessing on all labeled training records only**. Refitting is appropriate now because model selection is already complete: using the full labeled data gives the selected model more examples without using any hidden test labels.

The same imputation values, derived features, column order, and scaling rules are applied to every Kaggle test row. Fitting an imputer or scaler separately on the test set would make the training and prediction representations inconsistent.

In [ ]:
df_test = pd.read_csv(f"{DATA_DIR}/test.csv")
assert "id" in df_test.columns, "The submission requires the test-set id column."
assert set(X.columns) == set(df_test.columns) - {"id"}, (
    "Training/test feature names do not match. Inspect columns before submitting."
)
test_data = df_test.reindex(columns=X.columns)
print("Unlabeled test features:", test_data.shape)
display(test_data.head())

## 14. Refit preprocessing consistently on all labeled data

I now estimate **median imputation and continuous-feature scaling from the entire labeled `train.csv`**, since the holdout experiment has finished. No target values from `test.csv` are available or used. The geographic features are created with the exact same deterministic formulas as in Part 1.

In [ ]:
final_imputer = SimpleImputer(strategy="median")
X_full_imputed = pd.DataFrame(
    final_imputer.fit_transform(X), columns=X.columns, index=X.index
)
test_imputed = pd.DataFrame(
    final_imputer.transform(test_data), columns=X.columns, index=test_data.index
)

X_full_ready = add_geographic_features(X_full_imputed)
test_ready = add_geographic_features(test_imputed)
assert X_full_ready.columns.tolist() == test_ready.columns.tolist()

final_continuous_cols = [
    col for col in X_full_ready.columns if X_full_ready[col].nunique() > 2
]
final_scaler = StandardScaler()
X_full_scaled = X_full_ready.copy()
test_scaled = test_ready.copy()
X_full_scaled[final_continuous_cols] = final_scaler.fit_transform(
    X_full_ready[final_continuous_cols]
)
test_scaled[final_continuous_cols] = final_scaler.transform(
    test_ready[final_continuous_cols]
)
print("Prepared full training rows:", len(X_full_ready), "test rows:", len(test_ready))

## 15. Retrain the selected classifier and predict test labels

Only **one** of the five candidate classifiers is used for the final submission. I copy its chosen hyperparameters, retrain it on every labeled training row, and predict the unseen test classes. This final refit does **not** produce a new validation score; the score reported above belongs to the earlier 80/20 comparison, not to the new fit.

In [ ]:
selected_model = clone(models[best_name])
if best_name in scaled_models:
    selected_model.fit(X_full_scaled, y)
    test_pred = selected_model.predict(test_scaled)
else:
    selected_model.fit(X_full_ready, y)
    test_pred = selected_model.predict(test_ready)

print("Selected model:", best_name)
print("Test predictions generated:", len(test_pred))
print("Unique predicted labels:", np.unique(test_pred))

## 16. Create and validate the Kaggle submission CSV

The file must preserve the **original test-row IDs** and have the template's exact `id,label` column names. Assertions check the row count, unique IDs, missing predictions, and target labels before saving. Submitting this CSV to Kaggle will provide a separate leaderboard score on the competition's hidden labels.

In [ ]:
df_submission = pd.DataFrame({"id": df_test["id"], "label": test_pred})
assert df_submission.columns.tolist() == ["id", "label"]
assert len(df_submission) == len(df_test)
assert not df_submission["id"].isna().any()
assert not df_submission["label"].isna().any()
assert not df_submission["id"].duplicated().any()
assert set(df_submission["label"]).issubset(set(y))
df_submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(df_submission.head())

## 17. Final reflection and limitations

**Workflow:** Separating `id` and `label`, using a stratified holdout, learning imputation/scaling parameters only from training data, and applying consistent transformations help keep the evaluation meaningful. Derived geographic variables are reasonable candidate predictors, although their usefulness is determined by model performance rather than assumed.

**Evidence and decision:** The validation comparison above reports all five models with identical held-out rows, including accuracy, weighted F1, macro F1, and the training–validation gap. The **results-based discussion** identifies the actual winner and runner-up and justifies the model choice using observed scores, not intuition alone. The per-class report and confusion matrix expose limitations that the overall score might conceal.

**Uncertainty:** One holdout split can favor a model by chance. The test set is unlabeled, so its true score is unknown until Kaggle evaluates the submission. Additional cross-validation or careful hyperparameter tuning could improve confidence and performance, but would require a separate experiment rather than claiming the existing scores are more certain than they are.

**Final submission check:** After running *Save & Run All* in Kaggle, verify that the validation discussion shows actual numbers, `submission.csv` exists, and the CSV meets the competition's expected format. Report the **actual Kaggle leaderboard score** separately once it is available. If the competition's evaluation metric is accuracy, rerun with `PRIMARY_METRIC = "accuracy"` before the final model selection.